# IMRF 2026 Pre-conference Workshop
# Introduction to Python and AI-Based Image Analysis

### Part 2: Video Analysis

For this part we will use [MediaPipe library](https://developers.google.com/edge/mediapipe/solutions/guide). In particular, we will analyse a video dataset and extract hand landmarks using the [Hand Landmarker task](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker)

This cell installs the `mediapipe` library

In [ ]:
!pip install mediapipe

This cell downloads the pre-trained `hand_landmarker.task` model

In [ ]:
!wget https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

## Dataset

In this notebook, we use the [IPN Hand Dataset](https://gibranbenitez.github.io/IPN_Hand/)

> Benitez-Garcia, Gibran, et al. "Ipn hand: A video dataset and benchmark for real-time continuous hand gesture recognition." 2020 25th international conference on pattern recognition (ICPR). IEEE, 2021.

You can explore the different gestures inside the dataset [here](https://gibranbenitez.github.io/IPN_Hand/Classes)

In [ ]:
!gdown https://drive.google.com/drive/folders/1O4Fn_jbAEcKIksXHmQIMcHTyaDw4wt9x --folder

In [ ]:
!gdown https://drive.google.com/drive/folders/1-mihJEIFoNDpfo1puF8xAMJz6PGVKsBD --folder

This cell extracts the downloaded video archive files (`.tgz`) into the `videos/` directory, making the video files accessible for processing

In [ ]:
! tar -xzf videos/videos01.tgz
! tar -xzf videos/videos02.tgz
! tar -xzf videos/videos03.tgz
! tar -xzf videos/videos04.tgz
! tar -xzf videos/videos05.tgz

This cell lists the contents of the `videos` directory to confirm that the video files have been extracted correctly

In [ ]:
!ls videos

## Loading libraries and dataset

This cell imports all the necessary Python libraries:
- `pathlib` for path manipulation
- `tqdm` for progress bars
- `numpy` for numerical and array operations
- `scipy.io` for MATLAB file operations
- `pandas` for tabular data manipulation
- `cv2` for OpenCV video processing
- `matplotlib.pyplot` for plotting
- `mediapipe` for hand landmark detection

In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np
import scipy.io
import pandas as pd
import cv2
import matplotlib.pyplot as plt


import mediapipe as mp
from mediapipe.tasks import python

In [ ]:
#@markdown Run this cell to load some visualization functions for mediapipe <br/> Taken from [this notebook](https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/hand_landmarker/python/hand_landmarker.ipynb)

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54) # vibrant green

def draw_landmarks_on_image(rgb_image, detection_result):
  hand_landmarks_list = detection_result.hand_landmarks
  handedness_list = detection_result.handedness
  annotated_image = np.copy(rgb_image)

  # Loop through the detected hands to visualize.
  for idx in range(len(hand_landmarks_list)):
    hand_landmarks = hand_landmarks_list[idx]
    handedness = handedness_list[idx]

    # Draw the hand landmarks.
    mp_drawing.draw_landmarks(
      annotated_image,
      hand_landmarks,
      mp_hands.HAND_CONNECTIONS,
      mp_drawing_styles.get_default_hand_landmarks_style(),
      mp_drawing_styles.get_default_hand_connections_style())

    # Get the top left corner of the detected hand's bounding box.
    height, width, _ = annotated_image.shape
    x_coordinates = [landmark.x for landmark in hand_landmarks]
    y_coordinates = [landmark.y for landmark in hand_landmarks]
    text_x = int(min(x_coordinates) * width)
    text_y = int(min(y_coordinates) * height) - MARGIN

    # Draw handedness (left or right hand) on the image.
    cv2.putText(annotated_image, f"{handedness[0].category_name}",
                (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)

  return annotated_image

In [ ]:
# create the variables containing the input video path, the annotation list and output path

in_video_path=Path("videos/")
out_path=Path("output/")
annotation_file="annotations/Annot_List.txt"

In [ ]:
# create output folder

out_path.mkdir(exist_ok=True, parents=True)
# NOTE: if you want to make sure you don't overwrite an existing output folder, put exist_ok=False

Now we read the annotation list from the dataset.

We want to consider only gestures with label `B04: pointing with one finger`

Have a look at [IPN Hand Dataset](https://gibranbenitez.github.io/IPN_Hand/) and [List of gestures](https://gibranbenitez.github.io/IPN_Hand/Classes)

In [ ]:
# read the annotation list

df=pd.read_csv(annotation_file)
df

In [ ]:
# take only the portions of the videos with class B04
df_subset = df[df["label"]=="B0A"]

# take only the first occurence for each video
df_subset = df_subset.drop_duplicates(subset="video", keep="first")

df_subset

In [ ]:
# extract the ids of all videos

video_id_list=df_subset.video.unique().tolist()
video_id_list

## Process the videos to extract the hand landmarks

In [ ]:
# setting up mediapipe hand landmarker

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
    num_hands=1, # expecting 1 hands
    running_mode=VisionRunningMode.VIDEO) # set video mode

This cell defines the `process_video` function, which takes as input:
- `in_video`: the path of the input video
- `start_frame`: the initial frame of the video to process
- `end_frame`: the final frame of the video to process

The function returns `hand_landmarker_res` a list of mediapipe `HandLandmarkerResult` for each frame

In [ ]:
# function to process the video and extract the landmarks

def process_video(in_video, start_frame, end_frame):
  hand_landmarker_res=[]

  # open the video
  cap = cv2.VideoCapture(in_video)

  # check if video was opened successfully
  if not cap.isOpened():
      print("Error: Could not open video")
      return

  cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
  with HandLandmarker.create_from_options(options) as landmarker:
    # read frames in a loop
    while True:
        ret, frame = cap.read()

        # ret = False when no more frames (end of video)
        if not ret:
            break

        frame_idx = cap.get(cv2.CAP_PROP_POS_FRAMES)

        # finished the frames to process
        if frame_idx>=end_frame:
            break

        # extract timestamp in ms from opencv
        frame_timestamp=int(cap.get(cv2.CAP_PROP_POS_MSEC))

        # IMPORTANT: convert the opencv frame to RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # create a mediapipe Image
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        # run the hand detector
        hand_landmarker_res.append(landmarker.detect_for_video(mp_image, frame_timestamp))

    # close the video
    cap.release()

  return hand_landmarker_res

In [ ]:
results=[] # this is a list of all the videos

N=10 # number of samples to process

for video_id in tqdm(video_id_list[:N]): # iterate over the input videos
  in_video=in_video_path / (video_id + ".avi")

  sample=df_subset[df_subset["video"]==video_id] # extract the metadata of the video from df_subset

  res=process_video(in_video=in_video, start_frame=sample.t_start.item(), end_frame=sample.t_end.item())

  results.append(res)

Overal `results` is a list of list:

` #videos x #frame` of `HandLandmarkerResult`

In [ ]:
"number of videos", len(results)

In [ ]:
"number of frames in the first video", len(results[0])

Analyse the results of the `HandLandmarkerResult`: check [hand landmarks](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker#models) and [python reference](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python#handle_and_display_results)


In [ ]:
results[0][0]

This cell defines the `write_output_video` function, which takes as input:
- `in_video`: the path of the input video
- `start_frame`: the initial frame of the video to process
- `end_frame`: the final frame of the video to process
- `mp_results`: the list of `HandLandmarkerResult` for the frame
- `out_video`: the path of the output video


In [ ]:
def write_ouput_video(in_video, start_frame, end_frame, mp_results, out_video):

  # reopen the video
  cap = cv2.VideoCapture(in_video)

  # check if video was opened successfully
  if not cap.isOpened():
      print("Error: Could not open video")
      return

  fps = cap.get(cv2.CAP_PROP_FPS)
  width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

  # open output video
  fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # codec for .mp4
  out_cap = cv2.VideoWriter(out_video, fourcc, fps, (width, height))

  if not out_cap.isOpened():
    print("Error: Could not write video")
    return


  # read frames in a loop
  cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

  i=0

  while True:
      ret, frame = cap.read()

      # ret = False when no more frames (end of video)
      if not ret:
          break

      frame_idx = cap.get(cv2.CAP_PROP_POS_FRAMES)

      # finished the frames to process
      if frame_idx>=end_frame:
          break

      # extract timestamp in ms from opencv
      frame_timestamp=int(cap.get(cv2.CAP_PROP_POS_MSEC))

      # IMPORTANT: convert the opencv frame to RGB
      rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

      # draw landmarks on the video
      out_frame = draw_landmarks_on_image(rgb_frame, mp_results[i])

      # write frame on output video, IMPORTANT: convert frame back to BGR
      out_cap.write(cv2.cvtColor(out_frame, cv2.COLOR_RGB2BGR))
      i+=1

  # close the videos
  out_cap.release()
  cap.release()

## Explore the results on the first video

In [ ]:
# save the first video with landmarks
# we must reopen the input video, read frame by frame and save the new video

video_id=video_id_list[0]
mp_landmark_results=results[0]

sample=df_subset[df_subset["video"]==video_id]
in_video=in_video_path / (video_id + ".avi")
start_frame=sample.t_start.item()
end_frame=sample.t_end.item()

out_video=out_path / (video_id + ".mp4")

write_ouput_video(in_video, start_frame, end_frame, mp_landmark_results, out_video)

This cell defines the `convert_to_numpy_arrays` function, which takes as input:
- `hand_landmarker_results`: the list of results of the `HandLandmarker` for each frame of the video

It returns:
- `scores`: the reliability of the hand detection -> `# frames x 2 (left and right)`
- `coordinates`: the coordinates of the hand landmarks -> `# frames x 2 (left and right) x 21 x 3 (x,y,z)`

In [ ]:
def convert_to_numpy_arrays(hand_landmarker_results):
  # extract hand_landmarker_results into 3 numpy array

  num_frames = len(hand_landmarker_results)

  # each hand has 21 landmarks, each with 3 coordinates (x, y, z), if the hand is not present -> nan
  coordinates = np.full((num_frames, 2, 21, 3), np.nan) # shape: (num_frames, 2 hands, 21 landmarks, 3 xyz)

  # for each frame we have the accuracy score of the detection, if the hand is not present -> 0
  scores = np.zeros((num_frames, 2))

  for i, result in enumerate(hand_landmarker_results):
      if len(result.handedness)==0: # no hands detected, skip
          continue

      for idx, handedness in enumerate(result.handedness):
          category = handedness[0]
          landmarks = result.hand_landmarks[idx]

          # Extract (x, y, z) for all 21 landmarks
          coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])

          if category.category_name == 'Left': # index 0 corresponds to left
              coordinates[i, 0] = coords
              scores[i, 0] = category.score
          elif category.category_name == 'Right': # index 1 corresponds to right
              coordinates[i, 1] = coords
              scores[i, 1] = category.score

  return scores, coordinates

Extract the numpy arrays from `results` for the first video

In [ ]:
scores, coordinates = convert_to_numpy_arrays(results[0])

In [ ]:
scores.shape

Plot the `scores` variable

In [ ]:
plt.plot(scores[:,0]) # left
plt.plot(scores[:,1]) # right
plt.legend(["left", "right"])
plt.title("Scores")
plt.grid();

In [ ]:
# pick correct hand
hand_in_video=scores.mean(axis=0).argmax()

hand_in_video # 0 -> left, 1-> right

In [ ]:
# extract index position

MP_INDEX=8 # this is the id of the index tip in mediapipe

index_coord=coordinates[:, hand_in_video, MP_INDEX, :]

index_coord.shape

Plot the trajectory of the index finger

In [ ]:
# scatter
plt.scatter(index_coord[:,0], index_coord[:,1])
plt.xlim([0,1])
plt.ylim([1,0]) # inverted y axis (origin is in top left corner)
plt.title("Trajectory of index")
plt.xlabel("x")
plt.ylabel("y")
plt.grid();

This cell saves the `index_coord` NumPy array (containing the index finger tip coordinates) to a binary `.npy` file

In [ ]:
# save numpy

out_npy_file=out_path / (video_id + ".npy")
np.save(out_npy_file, index_coord)

This cell loads the `.npy` file that was just saved and then uses `np.testing.assert_array_equal` to verify that the loaded array is identical to the original `index_coord` array, ensuring data integrity

In [ ]:
# load the numpy array and check

index_coord_loaded=np.load(out_npy_file)
np.testing.assert_array_equal(index_coord_loaded, index_coord) # make sure the arrays are equals

This cell saves the `index_coord` NumPy array to a MATLAB `.mat` file. This format is often used for interoperability with MATLAB environments

In [ ]:
# save matlab

out_mat_file = out_path / (video_id + ".mat")
scipy.io.savemat(out_mat_file, {'index_coord': index_coord})

## Perfom some analyses on the first video

In [ ]:
# compute velocity
index_velocity=np.gradient(index_coord, axis=0)
index_coord.shape, index_velocity.shape

Plot the trajectory and velocity along the x-axis

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.plot(index_coord[:, 0], color='blue')
ax1.set_xlabel('frame')
ax1.set_ylabel("position (x)", color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(index_velocity[:, 0], color='red')
ax2.set_ylabel('velocity (x)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add a horizontal line at y=0 for the velocity axis
ax2.axhline(0, color='red', linestyle='--', linewidth=1.5)

plt.title("Position and velocity along x-axis over frames")
fig.tight_layout()
plt.grid()

## Homework - perform the analysis for all videos

1. process the `results` list in a for loop
2. compute the max velocity
3. write a csv file with `video_id` and `max_velocity`

### Solution

In [ ]:
#@markdown Here you find the solution

max_vel=[]

for i, res in enumerate(results):

    # convert results to numpy
    tmp_scores, tmp_coords = convert_to_numpy_arrays(res)

    # pick hands in the video (0: left, 1: right)
    hand_idx = tmp_scores.mean(axis=0).argmax()

    # extract index finger (MP_INDEX = 8)
    traj = tmp_coords[:, hand_idx, MP_INDEX, :]

    # compute velocity and its norm
    vel=np.gradient(traj, axis=0)
    norm_vel=np.linalg.norm(vel[~np.isnan(vel).any(axis=1)], axis=1)

    max_vel.append(np.max(norm_vel))

df_out=pd.DataFrame({"video_id": video_id_list[:N], "max_vel": max_vel})
df_out.to_csv(out_path / "max_velocities.csv")